In [3]:
import numpy as np

rng = np.random.default_rng(0)

In [4]:
def lu(A):
    U = np.array(A, dtype=float)
    n = U.shape[0]
    tol=1e-10

    L = np.eye(n)

    for k in range(n - 1):

        pivot = U[k, k]
        if abs(pivot) < tol:
            raise ValueError("zero pivot")
            
        for i in range(k + 1, n):
            factor = U[i, k] / pivot
            L[i, k] = factor

            for j in range(k, n):
                U[i, j] = U[i, j] - factor * U[k, j]

    return L, U

In [5]:
A = np.array([
    [2, 1, 1],
    [4, 3, 3],
    [8, 7, 9]
])

In [6]:
L, U = lu(A)
L

array([[1., 0., 0.],
       [2., 1., 0.],
       [4., 3., 1.]])

In [7]:
U

array([[2., 1., 1.],
       [0., 1., 1.],
       [0., 0., 2.]])

In [8]:
A = rng.normal(size=(6, 6))
A

array([[ 0.12573022, -0.13210486,  0.64042265,  0.10490012, -0.53566937,
         0.36159505],
       [ 1.30400005,  0.94708096, -0.70373524, -1.26542147, -0.62327446,
         0.04132598],
       [-2.32503077, -0.21879166, -1.24591095, -0.73226735, -0.54425898,
        -0.31630016],
       [ 0.41163054,  1.04251337, -0.12853466,  1.36646347, -0.66519467,
         0.35151007],
       [ 0.90347018,  0.0940123 , -0.74349925, -0.92172538, -0.45772583,
         0.22019512],
       [-1.00961818, -0.20917557, -0.15922501,  0.54084558,  0.21465912,
         0.35537271]])

In [9]:
L, U = lu(A)
L

array([[  1.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [ 10.37141296,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [-18.49221893,  -1.14867487,   1.        ,   0.        ,
          0.        ,   0.        ],
       [  3.27391881,   0.63655149,   1.13516006,   1.        ,
          0.        ,   0.        ],
       [  7.18578377,   0.45023801,  -0.94400393,  -0.48066137,
          1.        ,   0.        ],
       [ -8.03003586,  -0.54806882,   0.44344226,   0.17934338,
         -0.07664185,   1.        ]])

In [10]:
U

array([[ 0.12573022, -0.13210486,  0.64042265,  0.10490012, -0.53566937,
         0.36159505],
       [ 0.        ,  2.31719505, -7.34582302, -2.35338391,  4.93237382,
        -3.70892566],
       [ 0.        ,  0.        ,  2.15896265, -1.49570436, -4.78428047,
         2.11004508],
       [ 0.        ,  0.        ,  0.        ,  4.21894289,  3.37975755,
        -0.86663952],
       [ 0.        ,  0.        ,  0.        ,  0.        , -1.72112438,
         0.86708126],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.51244506]])

In [11]:
np.allclose(L @ U, A)

True

In [12]:
def solve_lu(L, U, b):
    n = len(b)
    y = np.zeros(n)
    x = np.zeros(n)

    for i in range(n):
        s = 0

        for j in range(i):
            s += L[i, j] * y[j]

        y[i] = (b[i] - s) / L[i, i]

    for i in range(n - 1, -1, -1):
        s = 0

        for j in range(i + 1, n):
            s += U[i, j] * x[j]

        x[i] = (y[i] - s) / U[i, i]

    return x

In [13]:
A = np.array([
    [1e-17, 1],
    [1, 1]
])
A

array([[1.e-17, 1.e+00],
       [1.e+00, 1.e+00]])

In [14]:
b = np.array([1, 2])
b

array([1, 2])

In [15]:
L, U = lu(A)
x = solve_lu(L, U, b)
x

ValueError: zero pivot

It fails because the first pivot is 10^-17, which is smaller than the tolerance 10^-10. Therefore, our function raises zero pivot error. The matrix is invertible, but the pivot is too small for our implementation. So we need to try swapping to see if it might change the solution

In [16]:
A2 = A[[1, 0]]
b2 = b[[1, 0]]

In [17]:
L2, U2 = lu(A2)
x2 = solve_lu(L2, U2, b2)
x2

array([1., 1.])

In [18]:
np.allclose(x2, np.linalg.solve(A2, b2))

True

After swapping the two rows of A and b, the first pivot becomes 1, and the solver returns the correct solution. So partial pivoting is indeed important for numerical stability.

In [19]:
def dual_basis(P, tol=1e-10):
    P = np.array(P, dtype=float)
    n = P.shape[0]
    tol=1e-10

    R = np.zeros((n, 2 * n))
    R[:, :n] = P

    for i in range(n):
        R[i, n + i] = 1

    for c in range(n):
        pivot_row = c

        for i in range(c + 1, n):
            if abs(R[i, c]) > abs(R[pivot_row, c]):
                pivot_row = i

        if abs(R[pivot_row, c]) < tol:
            raise ValueError("Matrix is not invertible")

        R[[c, pivot_row]] = R[[pivot_row, c]]

        pivot = R[c, c]
        R[c] = R[c] / pivot

        for i in range(n):
            if i != c:
                factor = R[i, c]
                R[i] = R[i] - factor * R[c]

    return R[:, n:]

In [20]:
def sherman_morrison(G_inv, x):
    v = G_inv @ x
    denominator = 1 + x @ v
    G_new_inv = G_inv - np.outer(v, v) / denominator

    return G_new_inv

In [22]:
X = rng.normal(size=(200, 50))

In [23]:
G = X.T @ X
G_inv = dual_basis(G)
G_inv

array([[ 6.10706286e-03, -2.69434261e-04,  2.33033401e-04, ...,
        -2.40773332e-04,  4.37917934e-05, -1.20803932e-04],
       [-2.69434261e-04,  7.47968258e-03,  1.53295684e-03, ...,
         8.37323333e-05, -1.15841566e-03,  2.80307071e-04],
       [ 2.33033401e-04,  1.53295684e-03,  6.49652545e-03, ...,
        -1.15019921e-04, -6.01071808e-04, -6.91057538e-06],
       ...,
       [-2.40773332e-04,  8.37323333e-05, -1.15019921e-04, ...,
         6.49215794e-03, -9.21862230e-05,  5.31776087e-04],
       [ 4.37917934e-05, -1.15841566e-03, -6.01071808e-04, ...,
        -9.21862230e-05,  6.93351516e-03,  2.00863415e-04],
       [-1.20803932e-04,  2.80307071e-04, -6.91057538e-06, ...,
         5.31776087e-04,  2.00863415e-04,  7.08365739e-03]])

In [25]:
rows2 = rng.normal(size=(100, 50))
rows2

array([[ 0.01940167, -0.6209096 ,  0.83117344, ..., -1.07732573,
         0.02901275, -0.19933018],
       [-1.37435065, -0.20571979,  1.37189891, ..., -1.11014768,
         1.20053259, -0.43501283],
       [-0.67329222,  0.76858248, -0.50534991, ...,  0.869351  ,
         0.65694501, -0.62448451],
       ...,
       [ 0.57773588,  0.21681157, -0.91602478, ...,  0.21231866,
        -0.45214648, -1.08751111],
       [ 0.71667128,  0.82334225,  0.80872285, ..., -2.14372332,
         0.48840721,  0.02099782],
       [ 0.62810598,  0.1399931 ,  0.76925361, ...,  0.20591619,
         0.35066407,  0.3492841 ]])

In [27]:
import time
start = time.perf_counter()

for x in rows2:
    G_inv = sherman_morrison(G_inv, x)

duration = time.perf_counter() - start

print(duration)

0.004008221440017223


In [32]:
X_final = np.concatenate((X, rows2), axis=0)
X_final

array([[-0.65382861, -0.12961363,  0.78397547, ..., -1.09114612,
        -1.27768017,  0.63041149],
       [ 0.58116581,  1.29455882, -0.75460579, ...,  1.32245747,
        -0.01391467,  1.04183976],
       [ 1.40226483,  1.15016564, -2.36530391, ..., -0.2003297 ,
        -0.43686833,  0.51984173],
       ...,
       [ 0.57773588,  0.21681157, -0.91602478, ...,  0.21231866,
        -0.45214648, -1.08751111],
       [ 0.71667128,  0.82334225,  0.80872285, ..., -2.14372332,
         0.48840721,  0.02099782],
       [ 0.62810598,  0.1399931 ,  0.76925361, ...,  0.20591619,
         0.35066407,  0.3492841 ]])

In [33]:
start = time.perf_counter()

G_final = X_final.T @ X_final
G_inv_direct = dual_basis(G_final)

duration2 = time.perf_counter() - start
duration2

0.02199670299887657

In [34]:
np.allclose(G_inv, G_inv_direct)

True

The Sherman–Morrison updates are computationally cheaper than recomputing the inverse after every new row because each update requires (O(n^2)) operations instead of approximately (O(n^3)) for a matrix inversion.